# LangSmith Evaluation with the FRAMES Benchmark

## Part 1: Setup & Document Selection

This lab evaluates multi-hop factual question answering using 20 examples from the `test` split of Google's FRAMES benchmark. FRAMES is introduced in Krishna et al. (2024), [*Fact, Fetch, and Reason: A Unified Evaluation of Retrieval-Augmented Generation*](https://arxiv.org/abs/2409.12941). The source split has 824 rows and includes prompts, reference answers, reasoning types, and supporting Wikipedia links.


In [27]:
%pip install python-dotenv langsmith openai datasets openevals


Note: you may need to restart the kernel to use updated packages.


In [28]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-custom-dataset-evaluation"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith endpoint loaded:", bool(os.getenv("LANGSMITH_ENDPOINT")))


OpenAI key loaded: True
LangSmith key loaded: True
LangSmith endpoint loaded: True


In [29]:
from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from datasets import load_dataset

client = Client()
openai_client = wrap_openai(OpenAI())

In [30]:
# Test
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Reply with a short greeting and a check emoji at the end.",
        }
    ],
)

answer = response.choices[0].message.content
print(answer)

Hello! Hope you're having a great day! ✅


## Part 2: Dataset Creation

#### Extract and Structure Your Examples

In [32]:
from datasets import load_dataset

frames_dataset = load_dataset(
    "google/frames-benchmark",
    split="test",
)
print(f"Loaded FRAMES test split with {len(frames_dataset)} rows")
print("Available columns:", frames_dataset.column_names)

Loaded FRAMES test split with 824 rows
Available columns: ['Unnamed: 0', 'Prompt', 'Answer', 'wikipedia_link_1', 'wikipedia_link_2', 'wikipedia_link_3', 'wikipedia_link_4', 'wikipedia_link_5', 'wikipedia_link_6', 'wikipedia_link_7', 'wikipedia_link_8', 'wikipedia_link_9', 'wikipedia_link_10', 'wikipedia_link_11+', 'reasoning_types', 'wiki_links']


In [33]:
# Structure 20 independent FRAMES prompt/reference-answer examples.
sample_count = min(20, len(frames_dataset))
if sample_count < 10:
    raise ValueError(f"FRAMES test split has only {sample_count} rows; at least 10 are required")

frames_examples = []
for row in frames_dataset.select(range(sample_count)):
    raw_question = row["Prompt"]
    raw_answer = row["Answer"]
    if not isinstance(raw_question, str) or not raw_question.strip():
        raise ValueError(f"FRAMES row {row.get('Unnamed: 0', 'unknown')} has an empty prompt")
    if not isinstance(raw_answer, str) or not raw_answer.strip():
        raise ValueError(f"FRAMES row {row.get('Unnamed: 0', 'unknown')} has an empty answer")
    question = raw_question.strip()
    answer = raw_answer.strip()
    frames_examples.append({
        "question": question,
        "answer": answer,
        "reasoning_types": row.get("reasoning_types") or "unspecified",
        "wiki_links": row.get("wiki_links") or "",
        "source": "google/frames-benchmark",
        "split": "test",
    })

if len({example["question"] for example in frames_examples}) != len(frames_examples):
    raise ValueError("Duplicate FRAMES prompts found in the selected examples")

print(f"Prepared {len(frames_examples)} FRAMES examples")
print("Reasoning types:", sorted({e["reasoning_types"] for e in frames_examples}))

Prepared 20 FRAMES examples
Reasoning types: ['Multiple constraints', 'Multiple constraints | Temporal reasoning', 'Numerical reasoning | Multiple constraints | Temporal reasoning', 'Numerical reasoning | Tabular reasoning | Multiple constraints', 'Numerical reasoning | Tabular reasoning | Multiple constraints | Temporal reasoning', 'Numerical reasoning | Tabular reasoning | Post processing | Temporal reasoning', 'Tabular reasoning | Multiple constraints', 'Tabular reasoning | Multiple constraints | Temporal reasoning', 'Temporal reasoning']


In [34]:
# Create the LangSmith FRAMES dataset and verify its examples.
DATASET_NAME = "FRAMES-Multi-Hop-QA-Evaluation"
DATASET_DESCRIPTION = (
    "A 20-example subset of the google/frames-benchmark test split for evaluating "
    "multi-hop factual question answering against reference answers."
)

matching_datasets = [d for d in client.list_datasets() if d.name == DATASET_NAME]
if matching_datasets:
    dataset = matching_datasets[0]
    print(f"Using existing dataset: {dataset.name} ({dataset.id})")
else:
    dataset = client.create_dataset(DATASET_NAME, description=DATASET_DESCRIPTION)
    client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": {"question": e["question"]},
                "outputs": {"answer": e["answer"]},
                "metadata": {
                    key: e[key]
                    for key in ("reasoning_types", "wiki_links", "source", "split")
                },
            }
            for e in frames_examples
        ],
    )
    print(f"Created dataset: {dataset.name} ({dataset.id})")

uploaded_examples = list(client.list_examples(dataset_id=dataset.id))
if len(uploaded_examples) != len(frames_examples):
    raise RuntimeError(
        f"Expected {len(frames_examples)} uploaded examples, found {len(uploaded_examples)}"
    )
if any(not e.inputs.get("question") or not e.outputs or not e.outputs.get("answer") for e in uploaded_examples):
    raise RuntimeError("An uploaded dataset example is missing its input or reference answer")
if any(not e.metadata or not e.metadata.get("reasoning_types") for e in uploaded_examples):
    raise RuntimeError("An uploaded dataset example is missing reasoning metadata")

print(f"Verified {len(uploaded_examples)} examples in {DATASET_NAME}")
print(f"Dataset ID: {dataset.id}")

Created dataset: FRAMES-Multi-Hop-QA-Evaluation (dc312e9b-cd59-4b89-9214-3e1ccd0d80d7)
Verified 20 examples in FRAMES-Multi-Hop-QA-Evaluation
Dataset ID: dc312e9b-cd59-4b89-9214-3e1ccd0d80d7


## Part 3: Evaluation Setup

This part defines the traced target function and the correctness evaluator used on the FRAMES examples.


### Step 5: Implement and Smoke-Test the Target

The target receives a FRAMES question and returns a concise answer. It uses `gpt-4o-mini` at temperature 0, validates its input, and lets API failures surface rather than hiding them.


In [ ]:
MODEL_NAME = "gpt-4o-mini"

@traceable(name="frames_target")
def target_function(inputs: dict) -> dict:
    question = inputs.get("question")
    if not isinstance(question, str) or not question.strip():
        raise ValueError("The target requires a non-empty 'question' string")

    response = openai_client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        max_tokens=512,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the question accurately and directly. "
                    "Do not invent facts; if uncertain, say so."
                ),
            },
            {"role": "user", "content": question.strip()},
        ],
    )
    answer_text = response.choices[0].message.content
    if not answer_text or not answer_text.strip():
        raise RuntimeError("The model returned an empty answer")
    return {"answer": answer_text.strip()}

smoke_test_results = []
for index, example in enumerate(frames_examples[:3], start=1):
    prediction = target_function({"question": example["question"]})
    smoke_test_results.append(prediction)
    print(f"Smoke test {index}: {prediction['answer'][:250]}")


### Step 6: Configure the Correctness Evaluator

The LLM-as-judge evaluator compares each generated answer with the FRAMES reference answer. It scores factual accuracy, completeness, and whether the answer addresses the question; wording differences alone should not count as errors. The judge is `gpt-4o-mini`.


In [ ]:
correctness_evaluator = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="openai:gpt-4o-mini",
    feedback_key="correctness",
)
print("Correctness evaluator ready:", callable(correctness_evaluator))


## Part 4: Run and Review the Evaluation

### Step 7: Execute the LangSmith Experiment

The run uses all 20 dataset examples, the traced target, and the correctness evaluator. Concurrency is limited to two requests to avoid unnecessary rate-limit pressure.


In [ ]:
from datetime import datetime, timezone

experiment_prefix = (
    "frames-gpt-4o-mini-"
    + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")
)
evaluation_results = client.evaluate(
    target_function,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix=experiment_prefix,
    description="FRAMES multi-hop QA, gpt-4o-mini, temperature 0",
    max_concurrency=2,
    blocking=True,
)
evaluation_df = evaluation_results.to_pandas()
print("Experiment:", evaluation_results.experiment_name)
print("Experiment URL:", evaluation_results.url)
print("Result rows:", len(evaluation_df))


### Step 8: Review Individual Results in LangSmith

Open the experiment URL printed above. Review high- and low-scoring examples, especially the judge feedback and whether the answer handles the question's multiple constraints. Record a screenshot or link in `lab_proof.md`.


In [ ]:
display_columns = [
    column
    for column in (
        "example_id",
        "inputs.question",
        "outputs.answer",
        "reference.answer",
        "feedback.correctness",
        "execution_time",
        "error",
    )
    if column in evaluation_df.columns
]
print(evaluation_df[display_columns].to_string(index=False, max_colwidth=180))


## Part 5: Analyze Results and Report

### Step 9: Calculate Metrics and Inspect Failure Patterns

The analysis reports score distribution, pass rate, performance by FRAMES reasoning type, and example-level evaluator feedback. It writes a compact score export that excludes question and answer text.


In [ ]:
import json
from pathlib import Path

import pandas as pd

if "feedback.correctness" not in evaluation_df.columns:
    raise RuntimeError("The evaluation results do not contain correctness feedback")

analysis_df = evaluation_df.copy()
analysis_df["score"] = pd.to_numeric(
    analysis_df["feedback.correctness"],
    errors="coerce",
)
if analysis_df["score"].isna().all():
    raise RuntimeError("No numeric correctness scores were returned")

metadata_by_id = {
    str(example.id): (example.metadata or {})
    for example in uploaded_examples
}
analysis_df["reasoning_types"] = analysis_df["example_id"].astype(str).map(
    lambda example_id: metadata_by_id.get(example_id, {}).get(
        "reasoning_types", "unknown"
    )
)
analysis_df["passed"] = analysis_df["score"].fillna(0).ge(0.5)

run_ids = [str(run_id) for run_id in analysis_df["id"].dropna()]
feedback_rows = list(
    client.list_feedback(
        run_ids=run_ids,
        feedback_key=["correctness"],
        limit=len(run_ids),
    )
)
comments_by_run = {
    str(feedback.run_id): feedback.comment
    for feedback in feedback_rows
    if feedback.comment
}
analysis_df["judge_comment"] = analysis_df["id"].astype(str).map(comments_by_run)

scored = analysis_df["score"].dropna()
pass_rate = float(analysis_df["passed"].mean())
category_summary = (
    analysis_df.groupby("reasoning_types", dropna=False)
    .agg(
        example_count=("score", "size"),
        mean_score=("score", "mean"),
        pass_rate=("passed", "mean"),
    )
    .reset_index()
)
valid_rows = analysis_df.dropna(subset=["score"])
best_row = valid_rows.loc[valid_rows["score"].idxmax()]
worst_row = valid_rows.loc[valid_rows["score"].idxmin()]

metrics = {
    "dataset": DATASET_NAME,
    "experiment": evaluation_results.experiment_name,
    "experiment_url": evaluation_results.url,
    "example_count": int(len(analysis_df)),
    "scored_example_count": int(len(scored)),
    "mean_score": float(scored.mean()),
    "median_score": float(scored.median()),
    "pass_rate_at_0_5": pass_rate,
    "failed_run_count": int(analysis_df["error"].notna().sum()),
    "best_example_id": str(best_row["example_id"]),
    "best_score": float(best_row["score"]),
    "worst_example_id": str(worst_row["example_id"]),
    "worst_score": float(worst_row["score"]),
}

score_export = analysis_df[
    [
        "example_id",
        "reasoning_types",
        "score",
        "passed",
        "execution_time",
        "error",
    ]
].copy()
score_export.to_csv("frames_evaluation_scores.csv", index=False)
Path("frames_evaluation_metrics.json").write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8",
)

print(json.dumps(metrics, indent=2))
print("\nPerformance by reasoning type:")
print(category_summary.to_string(index=False))
print("\nLowest-scoring judge feedback:")
print(worst_row.get("judge_comment") or "No written feedback returned")


### Step 10: Create the Evaluation Summary

The next cell writes the required short `evaluation_summary.md` from the measured run. Review its limitations and recommendations before submission; add your LangSmith UI screenshot or review notes to `lab_proof.md`.


In [ ]:
worst_comment = str(worst_row.get("judge_comment") or "No written judge feedback was returned.")
worst_comment = " ".join(worst_comment.split())
if len(worst_comment) > 240:
    worst_comment = worst_comment[:237] + "..."

summary_paragraph = (
    f"Evaluated {metrics['example_count']} FRAMES multi-hop QA examples with "
    f"{MODEL_NAME} (temperature 0), using {MODEL_NAME} as an LLM-as-judge for "
    f"correctness. The mean score was {metrics['mean_score']:.3f}, the median "
    f"was {metrics['median_score']:.3f}, and the pass rate at 0.5 was "
    f"{metrics['pass_rate_at_0_5']:.1%}. The lowest-scoring example was "
    f"{metrics['worst_example_id']} ({metrics['worst_score']:.3f}); evaluator "
    f"feedback: {worst_comment} The sample contains only 20 of the 824 test "
    "examples, and the target receives questions without retrieved Wikipedia "
    "article text, so results measure parametric answering rather than retrieval. "
    "Next, inspect low-scoring cases and test retrieval-augmented context before "
    "drawing conclusions about the full benchmark."
)

summary_markdown = (
    "# FRAMES Evaluation Summary\n\n"
    + summary_paragraph
    + "\n\n"
    + f"- Dataset: `{DATASET_NAME}` ({metrics['example_count']} examples)\n"
    + f"- Experiment: [{evaluation_results.experiment_name}]({evaluation_results.url})\n"
    + f"- Scores: mean {metrics['mean_score']:.3f}; median {metrics['median_score']:.3f}; "
    + f"pass rate {metrics['pass_rate_at_0_5']:.1%}\n"
    + "- Detailed per-example scores: `frames_evaluation_scores.csv`\n"
    + "- Aggregate metrics: `frames_evaluation_metrics.json`\n"
)
Path("evaluation_summary.md").write_text(summary_markdown, encoding="utf-8")
print("Wrote evaluation_summary.md")
